# Chronos: 24/7 After-Hours Information Pricing & Weekend Drift Engine
### Bitget AI Base Camp Hackathon S2 — Track 1: Alpha Factory
**Sub-Theme:** After-Hours Information Pricing | **Asset:** Tokenized US Equities ($rNVDA, $rTSLA, $rSPY)

---
## 1. Executive Summary & Thesis
Traditional US equities close on Friday at 4:00 PM EST and reopen Monday at 9:30 AM EST (a 128-hour weekly closure). However, **tokenized US stocks (rTokens) trade 24/7**.

During weekends, traditional market makers are dark. Retail flow and speculative sentiment dominate rToken order books, pushing prices far away from Friday's fundamental anchor.

**Chronos** models this dynamic:
1. Captures the **Friday 16:00 EST anchor price** ($P_{anchor}$).
2. Filters **Justified Macro Drift** (via 24/7 liquid benchmarks like BTC and Gold) from **Excess Retail Drift**.
3. Opens statistical counter-positions when excess drift breaches $|Z| \ge 2.0\sigma$.
4. Captures mean-reversion alpha as prices converge back to fair value at the **Monday 08:00–09:30 EST opening window**.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.fetcher import ChronosDataFetcher
from src.strategy import ChronosConfig, ChronosStrategy
from backtest.engine import BacktestEngine
from backtest.validation import ChronosValidator
from analytics.tear_sheet import TearSheetGenerator

print('✅ Core Chronos modules successfully imported.')

## 2. Ingest Continuous 24/7 Market Data
We load continuous hourly data for tokenized equities alongside the 24/7 macro benchmark (BTC-USD).

In [ ]:
fetcher = ChronosDataFetcher(cache_dir='../data/cache')
df = fetcher.fetch_historical_series(token_symbol='NVDA', macro_symbol='BTC-USD', period='120d', interval='1h')
print(f'Ingested {len(df)} 24/7 hourly candles ({len(df)//24} days).')
df[['token_close', 'macro_close']].tail()

## 3. Run Strategy & Backtest (Net of 0.05% Fees + 0.05% Slippage)
We initialize the Chronos alpha engine and execute bar-by-bar accounting.

In [ ]:
config = ChronosConfig(
    symbol='rNVDA',
    macro_benchmark='BTC-USD',
    z_entry_threshold=2.0,
    z_exit_threshold=0.4,
    stop_loss_pct=0.035,
    taker_fee_pct=0.0005,
    slippage_pct=0.0005
)

engine = BacktestEngine(config)
result = engine.run(df)

print('=== PERFORMANCE METRICS ===')
for k, v in result.metrics.items():
    print(f'{k:<26}: {v}')

## 4. Strict Walk-Forward Audit: In-Sample (60d) vs Out-of-Sample (35d+)
> **Anti-Overfitting Constraint:** The Bitget AI Hackathon S2 specifically mandates testing out-of-sample Sharpe decay:
> **Condition:** $OS \ge 0.5 \times IS$.

In [ ]:
validator = ChronosValidator(config)
is_res, oos_res, report = validator.validate_split(df, is_days=60, oos_days=35)
print(report.summary_table)

if report.passed_decay_threshold:
    print(f'\n✅ AUDIT PASSED: Decay ratio is {report.decay_ratio:.2f} >= 0.50.')
else:
    print(f'\n⚠️ AUDIT ALERT: Decay ratio {report.decay_ratio:.2f} below 0.50.')

## 5. Performance Tear Sheet & Visual Analytics

In [ ]:
generator = TearSheetGenerator(output_dir='../reports/figures')
paths = generator.generate_all(result, prefix='notebook')

# Render equity curve inline
plt.figure(figsize=(10, 4.5), dpi=150)
eq = result.equity_curve / result.equity_curve.iloc[0]
bm = result.signals_df['benchmark_equity'] / result.signals_df['benchmark_equity'].iloc[0]
plt.plot(eq.index, eq, label='Chronos Strategy (Net of Fees)', color='#00A389', lw=2)
plt.plot(bm.index, bm, label='Buy & Hold Benchmark', color='#888', lw=1.2, ls='--')
plt.title('Chronos Cumulative Equity Growth vs Benchmark', fontweight='bold')
plt.ylabel('Normalized Portfolio Value ($)')
plt.legend()
plt.show()